In [3]:
import pandas as pd
import time

start = time.time()

# 指定列名，header=None 表示第一行不是表头
col_names = ['user_id', 'item_id', 'category_id', 'behavior_type', 'timestamp']
df = pd.read_csv("UserBehavior.csv", names=col_names, header=None)

# 如果能运行到这一步，再统计
print(df['behavior_type'].value_counts())
print(f"Pandas耗时: {time.time() - start:.2f}秒")

behavior_type
pv      89716264
cart     5530446
fav      2888258
buy      2015839
Name: count, dtype: int64
Pandas耗时: 42.00秒


In [4]:
import polars as pl
import time

start = time.time()

col_names = ['user_id', 'item_id', 'category_id', 'behavior_type', 'timestamp']
lf = pl.scan_csv("UserBehavior.csv", has_header=False, new_columns=col_names)

result = (
    lf.group_by("behavior_type")
    .agg(pl.len().alias("count"))
    .collect()
)

print(result)
print(f"Polars耗时: {time.time() - start:.2f}秒")

shape: (4, 2)
┌───────────────┬──────────┐
│ behavior_type ┆ count    │
│ ---           ┆ ---      │
│ str           ┆ u32      │
╞═══════════════╪══════════╡
│ fav           ┆ 2888258  │
│ pv            ┆ 89716264 │
│ buy           ┆ 2015839  │
│ cart          ┆ 5530446  │
└───────────────┴──────────┘
Polars耗时: 6.19秒


In [25]:
import duckdb
import time

start = time.time()

con = duckdb.connect()

# 创建一个视图，方便后续使用
con.execute("""
    CREATE OR REPLACE VIEW user_behavior AS
    SELECT * FROM read_csv_auto('UserBehavior.csv', header=False,
        columns={'user_id': 'INT', 'item_id': 'INT', 'category_id': 'INT',
                 'behavior_type': 'VARCHAR', 'timestamp': 'INT'})
""")

result = con.execute("""
    SELECT behavior_type, COUNT(*) as count
    FROM user_behavior
    GROUP BY behavior_type
""").df()

print(result)
print(f"DuckDB耗时: {time.time() - start:.2f}秒")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  behavior_type     count
0           fav   2888258
1          cart   5530446
2            pv  89716264
3           buy   2015839
DuckDB耗时: 3.61秒


In [17]:
import duckdb
import time

start = time.time()

con = duckdb.connect()
result = con.execute("""
    SELECT 
        COUNT(*) AS total_rows,
        SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS user_id_null,
        SUM(CASE WHEN behavior_type IS NULL OR behavior_type = '' THEN 1 ELSE 0 END) AS behavior_type_null_or_empty
    FROM read_csv_auto('UserBehavior.csv', header=False,
        columns={'user_id': 'INT', 'item_id': 'INT', 'category_id': 'INT',
                 'behavior_type': 'VARCHAR', 'timestamp': 'INT'})
""").df()

print("【缺失值探查结果】")
print(f"总行数: {result['total_rows'][0]:,}")
print(f"user_id 为空的行数: {result['user_id_null'][0]:,}")
print(f"behavior_type 为空或空字符串的行数: {result['behavior_type_null_or_empty'][0]:,}")
print(f"耗时: {time.time() - start:.2f}秒")

【缺失值探查结果】
总行数: 100,150,807
user_id 为空的行数: 0.0
behavior_type 为空或空字符串的行数: 0.0
耗时: 1.93秒


In [18]:
import duckdb
import time

start = time.time()

con = duckdb.connect()
# 在SQL内部完成转换，直接返回日期对象
result = con.execute("""
    SELECT 
        MIN(timestamp) AS min_ts,
        MAX(timestamp) AS max_ts,
        MIN(to_timestamp(timestamp)) AS min_date,
        MAX(to_timestamp(timestamp)) AS max_date
    FROM read_csv_auto('UserBehavior.csv', header=False,
        columns={'user_id': 'INT', 'item_id': 'INT', 'category_id': 'INT',
                 'behavior_type': 'VARCHAR', 'timestamp': 'INT'})
""").df()

min_ts = result['min_ts'][0]
max_ts = result['max_ts'][0]
min_date = result['min_date'][0]  # 现在是 datetime 对象
max_date = result['max_date'][0]

print("【时间异常诊断】")
print(f"最小时间戳: {min_ts}  ->  {min_date}")
print(f"最大时间戳: {max_ts}  ->  {max_date}")

# 用年份判断是否存在1970年附近的异常
if min_date.year < 1971:
    print("⚠️  发现异常时间戳（可能为1970年附近），需在清洗时剔除！")
else:
    print("✅ 时间戳范围正常，无早期异常数据。")

print(f"耗时: {time.time() - start:.2f}秒")

【时间异常诊断】
最小时间戳: -2134949234  ->  1902-05-08 06:32:46+08:00
最大时间戳: 2122867355  ->  2037-04-09 13:22:35+08:00
⚠️  发现异常时间戳（可能为1970年附近），需在清洗时剔除！
耗时: 1.96秒


In [19]:
import duckdb
import time

start = time.time()

con = duckdb.connect()
result = con.execute("""
    SELECT COUNT(DISTINCT user_id) AS uv
    FROM read_csv_auto('UserBehavior.csv', header=False,
        columns={'user_id': 'INT', 'item_id': 'INT', 'category_id': 'INT',
                 'behavior_type': 'VARCHAR', 'timestamp': 'INT'})
""").df()

print("【独立访客统计】")
print(f"独立访客数（UV）: {result['uv'][0]:,}")
print(f"耗时: {time.time() - start:.2f}秒")

【独立访客统计】
独立访客数（UV）: 987,994
耗时: 1.93秒


In [20]:
import duckdb
import time

start = time.time()

con = duckdb.connect()
query = """
    SELECT 
        EXTRACT(HOUR FROM to_timestamp(timestamp)) AS hour,
        COUNT(CASE WHEN behavior_type = 'pv' THEN 1 END) AS pv_count,
        COUNT(CASE WHEN behavior_type = 'buy' THEN 1 END) AS buy_count
    FROM read_csv_auto('UserBehavior.csv', header=False,
        columns={'user_id': 'INT', 'item_id': 'INT', 'category_id': 'INT',
                 'behavior_type': 'VARCHAR', 'timestamp': 'INT'})
    GROUP BY hour
    ORDER BY hour
"""

result = con.execute(query).df()
print("【24小时行为分布】")
print(result)

# 找出购买高峰小时
peak_hour = result.loc[result['buy_count'].idxmax(), 'hour']
peak_buy = result['buy_count'].max()
print(f"\n购买高峰时段: {int(peak_hour)} 时（购买次数 {peak_buy:,}）")
print(f"耗时: {time.time() - start:.2f}秒")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

【24小时行为分布】
    hour  pv_count  buy_count
0      0   3058122      57776
1      1   1422183      23169
2      2    769514      12012
3      3    525228       8026
4      4    449898       6748
5      5    581878       8135
6      6   1226989      18014
7      7   2229521      37679
8      8   3043136      64917
9      9   3729407      96134
10    10   4335950     127933
11    11   4214991     122048
12    12   4257521     118591
13    13   4655733     123427
14    14   4643740     122172
15    15   4808739     122733
16    16   4609983     116446
17    17   4205920     101301
18    18   4316184      95907
19    19   5434314     115032
20    20   6590521     133863
21    21   7544179     145433
22    22   7450308     138268
23    23   5612305     100075

购买高峰时段: 21 时（购买次数 145,433）
耗时: 2.72秒


In [22]:
import duckdb
import time
import os

start_time = time.time()

# 连接 DuckDB
con = duckdb.connect()

# 1. 创建临时视图，避免重复解析 CSV
con.execute("""
    CREATE OR REPLACE TEMP VIEW raw_data AS
    SELECT *
    FROM read_csv_auto('UserBehavior.csv', header=False,
        columns={'user_id': 'INT', 'item_id': 'INT', 'category_id': 'INT',
                 'behavior_type': 'VARCHAR', 'timestamp': 'INT'})
""")

# 2. 清洗：剔除 behavior_type 为空 和 时间戳 <= 0 的行（负数时间戳为脏数据）
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW cleaned_data AS
    SELECT *
    FROM raw_data
    WHERE behavior_type IS NOT NULL 
      AND behavior_type != ''
      AND timestamp > 0
""")

# 3. 分区导出为 Parquet（按 behavior_type 分区）
con.execute("""
    COPY (SELECT * FROM cleaned_data) 
    TO 'clean_data_partitioned' 
    (FORMAT PARQUET, PARTITION_BY (behavior_type), COMPRESSION 'snappy')
""")

end_time = time.time()
print(f"✅ 清洗分区完成，总耗时: {end_time - start_time:.2f} 秒")

# 4. 计算文件体积对比
def get_folder_size(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            if os.path.isfile(fp):
                total += os.path.getsize(fp)
    return total

original_size = os.path.getsize("UserBehavior.csv")
partitioned_size = get_folder_size("clean_data_partitioned")

print("\n【文件体积对比】")
print(f"原始 CSV 大小: {original_size / (1024**3):.2f} GB")
print(f"分区后 Parquet 总大小: {partitioned_size / (1024**3):.2f} GB")
print(f"压缩比: {original_size / partitioned_size:.2f}x")
print(f"空间节省: {(1 - partitioned_size/original_size)*100:.2f}%")

# 5. 展示分区目录结构（可选）
print("\n【分区目录结构】")
for root, dirs, files in os.walk("clean_data_partitioned"):
    level = root.replace("clean_data_partitioned", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files[:3]:  # 每个目录只显示前3个文件，避免输出过多
        print(f"{indent}  {file}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ 清洗分区完成，总耗时: 4.81 秒

【文件体积对比】
原始 CSV 大小: 3.42 GB
分区后 Parquet 总大小: 0.90 GB
压缩比: 3.80x
空间节省: 73.69%

【分区目录结构】
clean_data_partitioned/
  behavior_type=buy/
    data_0.parquet
  behavior_type=cart/
    data_0.parquet
  behavior_type=fav/
    data_0.parquet
  behavior_type=pv/
    data_0.parquet
